In [11]:
# -*- coding: utf-8 -*-
"""s1_reference_quality.py

S1 — Reference-based quality (ROUGE-L + BERTScore vs gold) for every output
of the demo_sensitivity_runner.py JSONLs: zero-shot + the K=5 random-draw
seeds. Gold reference is taken from the 'reference' field the runner saved
in each record — no external gold files needed.

Feeds E1 (strategy deltas at instance level) and E5 (consistency != quality):
    per-instance quality under the random-draw condition
        = mean ROUGE-L / BERTScore over the 5 seed outputs vs gold
    join per_instance CSV with s3_*_per_instance.csv on (task, instance_id)
    -> scatter S1 quality (x) vs nli_composite_s (y)
    -> "reliably incorrect" quadrant = high S3, low S1.

Rules (matching s3_nli_consistency.py):
  - everything per output first, then per instance, aggregate after
  - empty/whitespace outputs get NaN scores and are excluded from instance
    means (full UNKNOWN handling = S4, applied downstream)
  - instances with an empty gold reference are excluded entirely

Outputs (in OUT_DIR):
  s1_{MODEL_SLUG}_per_output.csv    — one row per task x condition x instance
  s1_{MODEL_SLUG}_per_instance.csv  — one row per task x instance:
        zero_rougeL, zero_bertscore,
        seed_mean/std/min/max_rougeL, seed_mean_bertscore, n_valid_seed_outputs
  s1_{MODEL_SLUG}_summary.csv       — one row per task x condition

Run:
    pip install rouge-score bert-score pandas torch
    python s1_reference_quality.py
"""

import os
import json
import numpy as np
import pandas as pd
import torch

# ------------------------------------------------------------------------
# Config
# ------------------------------------------------------------------------

MODEL_NAME = "epfl-llm/meditron-7b"   # <- must match the runner's MODEL_NAME
MODEL_SLUG = MODEL_NAME.split('/')[-1].lower().replace('-', '_').replace('.', '_')

OUT_DIR = '/workspace/demo_sensitivity_runs'

# Reference-based BERTScore: fixed checkpoint, RAW (not baseline-rescaled) —
# matches the scale of the existing experiments_extracted sheet (~0.65-0.90).
BERT_MODEL = 'distilbert-base-uncased'
BERT_RESCALE = False
BERT_BATCH = 64
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

with open(os.path.join(OUT_DIR, 'config_frozen.json')) as f:
    CFG = json.load(f)
TASKS, SEEDS = CFG['tasks'], CFG['seeds']

CONDITIONS = ['zero'] + [f'random_seed{s}' for s in SEEDS]

# ------------------------------------------------------------------------
# Load runner outputs
# ------------------------------------------------------------------------

def load_jsonl(path):
    with open(path, encoding='utf-8') as f:
        return [json.loads(l) for l in f if l.strip()]


def load_task(task):
    """{condition: {instance_id: (output_text, reference)}}; None if any file missing."""
    per_cond = {}
    for cond in CONDITIONS:
        path = os.path.join(OUT_DIR, f"gen_{MODEL_SLUG}_{task}_{cond}.jsonl")
        if not os.path.exists(path):
            print(f"  [skip] missing: {os.path.basename(path)}")
            return None
        per_cond[cond] = {r['instance_id']: (r['output_text'], r.get('reference', ''))
                          for r in load_jsonl(path)}
    return per_cond

# ------------------------------------------------------------------------
# Scorers
# ------------------------------------------------------------------------

from rouge_score import rouge_scorer
from bert_score import score as bert_score

_rouge = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)


def rouge_l(cand, ref):
    return _rouge.score(ref, cand)['rougeL'].fmeasure

# ------------------------------------------------------------------------
# Score
# ------------------------------------------------------------------------

per_output_rows = []

for task in TASKS:
    print(f"\n=== {task} ===")
    per_cond = load_task(task)
    if per_cond is None:
        continue

    instance_ids = sorted(set.intersection(*(set(d) for d in per_cond.values())))

    # gold reference per instance (identical across conditions; take zero's)
    refs = {iid: per_cond['zero'][iid][1] for iid in instance_ids}
    n_no_ref = sum(1 for iid in instance_ids if not refs[iid].strip())
    if n_no_ref:
        print(f"  [WARN] {n_no_ref} instances with empty gold reference — excluded")
    instance_ids = [iid for iid in instance_ids if refs[iid].strip()]
    print(f"  {len(instance_ids)} instances x {len(CONDITIONS)} conditions")

    for cond in CONDITIONS:
        cands, crefs, valid_iids, empty_iids = [], [], [], []
        for iid in instance_ids:
            out = per_cond[cond][iid][0]
            if out and out.strip():
                cands.append(out); crefs.append(refs[iid]); valid_iids.append(iid)
            else:
                empty_iids.append(iid)

        # BERTScore in one batched call per condition
        if cands:
            _, _, F = bert_score(cands, crefs, model_type=BERT_MODEL,
                                 lang='en', rescale_with_baseline=BERT_RESCALE,
                                 batch_size=BERT_BATCH, device=DEVICE, verbose=False)
            F = F.numpy()
        else:
            F = np.array([])

        for iid, cand, ref, f1 in zip(valid_iids, cands, crefs, F):
            per_output_rows.append({
                'model': MODEL_NAME, 'task': task, 'condition': cond,
                'instance_id': iid, 'is_empty': False,
                'rougeL': rouge_l(cand, ref), 'bertscore_f1': float(f1),
            })
        for iid in empty_iids:
            per_output_rows.append({
                'model': MODEL_NAME, 'task': task, 'condition': cond,
                'instance_id': iid, 'is_empty': True,
                'rougeL': np.nan, 'bertscore_f1': np.nan,
            })
        print(f"  [done] {cond}: {len(cands)} scored, {len(empty_iids)} empty")

# ------------------------------------------------------------------------
# Aggregate + save
# ------------------------------------------------------------------------

po = pd.DataFrame(per_output_rows)
po.to_csv(os.path.join(OUT_DIR, f's1_{MODEL_SLUG}_per_output.csv'), index=False)

# per instance: zero-shot scores + stats over the 5 seed outputs (E5 quality)
seed_conds = [c for c in CONDITIONS if c != 'zero']
zero = (po[po.condition == 'zero']
        .set_index(['task', 'instance_id'])[['rougeL', 'bertscore_f1']]
        .rename(columns={'rougeL': 'zero_rougeL', 'bertscore_f1': 'zero_bertscore'}))
seeds = (po[po.condition.isin(seed_conds)]
         .groupby(['task', 'instance_id'])
         .agg(n_valid_seed_outputs=('rougeL', lambda s: int(s.notna().sum())),
              seed_mean_rougeL=('rougeL', 'mean'),
              seed_std_rougeL=('rougeL', 'std'),
              seed_min_rougeL=('rougeL', 'min'),
              seed_max_rougeL=('rougeL', 'max'),
              seed_mean_bertscore=('bertscore_f1', 'mean')))
pi = zero.join(seeds, how='outer').reset_index()
pi.insert(0, 'model', MODEL_NAME)
pi['seed_rougeL_spread'] = pi.seed_max_rougeL - pi.seed_min_rougeL  # draw sensitivity
per_inst_csv = os.path.join(OUT_DIR, f's1_{MODEL_SLUG}_per_instance.csv')
pi.to_csv(per_inst_csv, index=False)

summary = (po.groupby(['task', 'condition'])
             .agg(n_instances=('instance_id', 'count'),
                  n_empty=('is_empty', 'sum'),
                  mean_rougeL=('rougeL', 'mean'),
                  median_rougeL=('rougeL', 'median'),
                  mean_bertscore=('bertscore_f1', 'mean'))
             .round(4)
             .reset_index())
summary.insert(0, 'model', MODEL_NAME)
summary['bert_model'] = BERT_MODEL
summary['bert_rescaled'] = BERT_RESCALE
summary_csv = os.path.join(OUT_DIR, f's1_{MODEL_SLUG}_summary.csv')
summary.to_csv(summary_csv, index=False)

print(f"\n### S1 summary — {MODEL_NAME} ###")
print(summary.to_string(index=False))
print(f"\nSaved -> {per_inst_csv}")
print(f"      -> {summary_csv}")

# E5 hint
print("\nE5 join: merge s1_..._per_instance.csv (seed_mean_rougeL) with "
      "s3_..._per_instance.csv (nli_composite_s) on (task, instance_id).")



=== aci_bench ===
  100 instances x 6 conditions
  [done] zero: 100 scored, 0 empty
  [done] random_seed0: 100 scored, 0 empty
  [done] random_seed1: 100 scored, 0 empty
  [done] random_seed2: 100 scored, 0 empty
  [done] random_seed3: 100 scored, 0 empty
  [done] random_seed4: 100 scored, 0 empty

=== meddialog ===
  500 instances x 6 conditions
  [done] zero: 498 scored, 2 empty
  [done] random_seed0: 500 scored, 0 empty
  [done] random_seed1: 500 scored, 0 empty
  [done] random_seed2: 500 scored, 0 empty
  [done] random_seed3: 500 scored, 0 empty
  [done] random_seed4: 500 scored, 0 empty

=== medicationqa ===
  [WARN] 1 instances with empty gold reference — excluded
  499 instances x 6 conditions
  [done] zero: 499 scored, 0 empty
  [done] random_seed0: 499 scored, 0 empty
  [done] random_seed1: 499 scored, 0 empty
  [done] random_seed2: 499 scored, 0 empty
  [done] random_seed3: 499 scored, 0 empty
  [done] random_seed4: 499 scored, 0 empty

=== mtsamples ===
  500 instances x 6 